# Switch to the 'exploration' schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS exploration;

USE CATALOG `operations-catalog`;
USE SCHEMA exploration;

SELECT current_schema() AS schema, current_catalog() AS catalog;

# Inspection

In [0]:
%sql DESCRIBE bronze_purchase_order_flat

In [0]:
%sql DESCRIBE TABLE EXTENDED bronze_purchase_order_flat;

# Examine columns

### 1. PO_ID

In [0]:
%sql
select PO_ID from bronze_purchase_order_flat
limit 3;

In [0]:
%sql
select PO_ID from bronze_purchase_order_flat
where PO_ID not like 'PO-%';

In [0]:
%sql
-- check for uniqueness of PO_ID
select count(*) as count, PO_ID
from bronze_purchase_order_flat
group by PO_ID
having count(*) > 1

### 2. SupplierID

In [0]:
%sql
select SupplierID from bronze_purchase_order_flat limit 3;

In [0]:
%sql
--check for nulls in SupplierID
select distinct(SupplierID) 
from bronze_purchase_order_flat 
where SupplierID not like 'SUP%' or SupplierID is null;

Contains Nulls.

### 3. OrderDate and ExpectedDeliveryDate


In [0]:
%sql
with casted_purchase_order as (
    select try_cast(OrderDate AS string) as OrdDate, try_cast(ExpectedDeliveryDate as string) as ExpectedDeliveryDate
    from bronze_purchase_order_flat
)
select * 
from casted_purchase_order
where
    OrdDate is null or 
    ExpectedDeliveryDate is null or
    OrdDate NOT RLIKE r'^\d{4}-\d{2}-\d{2}$' or
    ExpectedDeliveryDate NOT RLIKE r'^\d{4}-\d{2}-\d{2}$'

In [0]:
%sql
with casted_purchase_order as (
    select try_cast(OrderDate AS string) as OrdDate, try_cast(ExpectedDeliveryDate as string) as expDeliveryDate
    from bronze_purchase_order_flat
)
select OrdDate from casted_purchase_order
where OrdDate not rlike r'^\d{4}-\d{2}-\d{2}$'
or OrdDate is null;

In [0]:
%sql
with casted_purchase_order as (
    select try_cast(OrderDate AS string) as OrdDate, try_cast(ExpectedDeliveryDate as string) as expDeliveryDate
    from bronze_purchase_order_flat
)
select (
    select count(*) from casted_purchase_order
    where OrdDate is null or
    OrdDate NOT RLIKE r'^\d{4}-\d{2}-\d{2}$'
) as nulls_in_ord_date,
(
    select count(*) from casted_purchase_order
    where expDeliveryDate is null or
    expDeliveryDate NOT RLIKE r'^\d{4}-\d{2}-\d{2}$'
) as nulls_in_expected_delivery_date,
count(*) as total_records,
nulls_in_ord_date/total_records as nulls_in_ord_date_percentage,
nulls_in_expected_delivery_date/total_records as nulls_in_expected_delivery_date_percentage

from casted_purchase_order

### 4. Status

In [0]:
%sql
select distinct(Status) from bronze_purchase_order_flat;

In [0]:
%sql
select count(*) as count, Status
from bronze_purchase_order_flat
group by Status


Databricks visualization. Run in Databricks to view.

### 5. Total Cost

In [0]:
%sql
select TotalCost from bronze_purchase_order_flat limit 5;

In [0]:
%sql
select try_cast(TotalCost as DECIMAL(8, 2)) from bronze_purchase_order_flat limit 5;

In [0]:
%sql
select count(*) as nulls_count, count(*)/(select count(*) from bronze_purchase_order_flat) as nulls_pct
from (select try_cast(TotalCost as DECIMAL(8, 2)) as totalcost from bronze_purchase_order_flat)
where totalcost is null